# 🎬 Netflix Content Strategy Analysis

This notebook provides a detailed exploratory data analysis (EDA) of the Netflix titles dataset. We explore content trends, genre distributions, and production footprints to understand the strategic direction of the streaming giant's library.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import ast
import os

# Set Plotly template for dark theme
pio.templates.default = "plotly_dark"

## 1. Data Loading & Cleaning

We load the `titles.csv` dataset and perform initial cleaning, focusing on parsing list-like columns (genres and production countries).

In [ ]:
def parse_list_col(val):
    try:
        if isinstance(val, str) and val.startswith('['):
            return ast.literal_eval(val)
        return val
    except:
        return val

df = pd.read_csv('titles.csv')

# Rename for clarity
df = df.rename(columns={'production_countries': 'country', 'genres': 'listed_in'})

# Parse lists
for col in ['country', 'listed_in']:
    if col in df.columns:
        df[col] = df[col].apply(parse_list_col)

# Extract year added if available, otherwise use release_year
if 'date_added' in df.columns:
    df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')
    df['year_added'] = df['date_added'].dt.year
else:
    df['year_added'] = df['release_year']

print(f"Dataset Loaded: {len(df)} titles.")
df.head()

## 2. High-Level Metrics

Analyzing the balance between Movies and TV Shows.

In [ ]:
content_counts = df['type'].value_counts().reset_index()
content_counts.columns = ['Type', 'Count']

fig_pie = px.pie(
    content_counts, 
    values='Count', 
    names='Type', 
    title="Library Composition: Movies vs TV Shows",
    color_discrete_sequence=['#E50914', '#444444']
)
fig_pie.show()

## 3. Maturity Profile

Understanding the distribution of age certifications.

In [ ]:
rating_counts = df['age_certification'].value_counts().reset_index()
rating_counts.columns = ['Rating', 'Count']

fig_rating = px.bar(
    rating_counts, 
    x='Rating', 
    y='Count', 
    title="Content Maturity Profile",
    color_discrete_sequence=['#E50914']
)
fig_rating.show()

## 4. Genre Composition

Deep dive into the genres using a Sunburst chart.

In [ ]:
sun_df = df[['type', 'listed_in']].explode('listed_in')
top_type_genres = sun_df.groupby(['type', 'listed_in']).size().reset_index(name='count')
top_type_genres = top_type_genres.sort_values(['type', 'count'], ascending=[True, False]).groupby('type').head(8)

fig_sun = px.sunburst(
    top_type_genres, 
    path=['type', 'listed_in'], 
    values='count',
    color='type', 
    color_discrete_map={'MOVIE': '#E50914', 'SHOW': '#444444'},
    title="Genre Distribution by Content Type"
)
fig_sun.show()

## 5. Acquisition Trends

How fast is the Netflix library growing?

In [ ]:
trend_df = df.groupby('year_added').size().reset_index(name='count')
trend_df = trend_df[trend_df['year_added'] > 2000] # Focus on modern era
trend_df.columns = ['Year', 'Titles Added']

fig_trend = px.area(
    trend_df, 
    x='Year', 
    y='Titles Added', 
    title="Library Growth Over Time",
    color_discrete_sequence=['#E50914']
)
fig_trend.show()

## 6. Global Production Hubs

Identifying the top countries contributing content.

In [ ]:
country_counts = df['country'].explode().value_counts().head(10).reset_index()
country_counts.columns = ['Country', 'Count']

fig_country = px.bar(
    country_counts, 
    x='Count', 
    y='Country', 
    orientation='h', 
    title="Top 10 Global Production Hubs",
    color_discrete_sequence=['#E50914']
)
fig_country.update_layout(yaxis={'categoryorder':'total ascending'})
fig_country.show()